# Revisión integral del paper — new plots

Notebook reproducible para respuestas a revisores. Todos los outputs se guardan en `new_plots_review/`.

## Secciones
1. Configuración
2. Fig. 2 (hora local, modularity/nestedness dual axis)
3. Suavizado (MA, mediana, Savitzky–Golay)
4. Comunidades (Q fija, AMI, entropía)
5. ε² canónico + nulos
6. Non-CTW + D-Mercator + navegabilidad
7. Matriz de respuestas


In [1]:
import sys
from pathlib import Path

REVIEW_ROOT = Path.cwd() / "new_plots_review" if (Path.cwd() / "new_plots_review").exists() else Path.cwd()
if REVIEW_ROOT.name != "new_plots_review":
    REVIEW_ROOT = Path.cwd()
REPO_ROOT = REVIEW_ROOT.parent if REVIEW_ROOT.name == "new_plots_review" else REVIEW_ROOT

sys.path.insert(0, str(REVIEW_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from config import ensure_dirs, MANIFESTATIONS, REVIEW_ROOT, RESULTS_DIR, FIGURES_DIR
ensure_dirs()
print("Review root:", REVIEW_ROOT)
print("Manifestations:", MANIFESTATIONS)


Review root: /home/msuarez/Escritorio/self_similarity/main_branch/github/self-similarity-in-social-movements/new_plots_review
Manifestations: ['nat', '9n', 'ch']


In [2]:
# 2. Fig. 2 — hora local
from temporal_plots import plot_fig2_panel, plot_fig2_combined, smoothing_comparison, plot_smoothing_comparison

for m in MANIFESTATIONS:
    plot_fig2_panel(m)
    smoothing_comparison(m)
    plot_smoothing_comparison(m)
plot_fig2_combined()


PosixPath('/home/msuarez/Escritorio/self_similarity/main_branch/github/self-similarity-in-social-movements/new_plots_review/figures/fig2_combined_local_time.png')

In [3]:
# 3. Comunidades y entropía de ventana
from community_analysis import compute_fixed_modularity_series, plot_community_comparison, window_entropy_diagnostic

window_entropy_diagnostic()
for m in MANIFESTATIONS:
    compute_fixed_modularity_series(m)
    plot_community_comparison(m)


In [4]:
# 4. ε² canónico y modelos nulos
from null_models import (
    ct_epsilon_summary,
    epsilon_regression_check,
    plot_epsilon_collapse_panels,
    plot_epsilon_collapse_temporal_combined,
    null_model_analysis,
    plot_null_model_summary,
)

ct_epsilon_summary()
epsilon_regression_check()
for m in MANIFESTATIONS:
    plot_epsilon_collapse_panels(m)
plot_epsilon_collapse_temporal_combined()
for m in MANIFESTATIONS:
    null_model_analysis(m, n_replicates=10)
    plot_null_model_summary(m)


In [5]:
# 5. Non-CTW, embeddings y navegabilidad
import pandas as pd
from pathlib import Path
from dmercator_utils import select_all_nonctw, ensure_embeddings
from navigability_utils import evaluate_embedding, parse_embedding_metadata, plot_navigability_comparison

nonctw = select_all_nonctw()
display(nonctw)
inventory = ensure_embeddings(nonctw)
display(inventory)

nav_rows = []
for _, row in inventory.iterrows():
    if str(row['source']).startswith('failed'):
        continue
    folder = Path(row['embedding_dir'])
    graph_id = str(row['graph_id'])
    if not (folder / f'{graph_id}.inf_coord').exists():
        continue
    stats = evaluate_embedding(folder, graph_id, label=f"{row['manifestacion']}_{row['window_type']}", window_type=row['window_type'].replace('CTW_sensitivity','CTW'))
    stats.update(parse_embedding_metadata(folder, graph_id))
    stats['manifestacion'] = row['manifestacion']
    nav_rows.append(stats)

nav_df = pd.DataFrame(nav_rows)
nav_df.to_csv(RESULTS_DIR / 'navigability_summary.csv', index=False)
if not nav_df.empty:
    plot_navigability_comparison(nav_df)
display(nav_df)


,manifestacion,ctw_hour,nonctw_hour,hour_window,ctw_hashtags,nonctw_hashtags,activity_ratio,activity_tolerance_used,epsilon_cco,selection_rule
0,nat,429624,429600,1,642,459.0,0.714953,0.3,0.5168,max epsilon_cco outside ±12h with matched acti...
1,9n,437037,437075,2,1327,1130.0,0.851545,0.2,0.7252,max epsilon_cco outside ±12h with matched acti...
2,ch,394717,394701,2,894,843.0,0.942953,0.2,0.5286,max epsilon_cco outside ±12h with matched acti...


,manifestacion,hour,window_type,embedding_dir,graph_id,source
0,nat,429624,CTW,/home/msuarez/Escritorio/self_similarity/main_...,429624,reused_existing
1,9n,437037,CTW,/home/msuarez/Escritorio/self_similarity/main_...,437037,reused_existing
2,ch,394717,CTW,/home/msuarez/Escritorio/self_similarity/main_...,394717,cached
3,ch,394718,CTW_sensitivity,/home/msuarez/Escritorio/self_similarity/main_...,394718,reused_existing
4,nat,429600,non-CTW,/home/msuarez/Escritorio/self_similarity/main_...,429600,cached
5,9n,437075,non-CTW,/home/msuarez/Escritorio/self_similarity/main_...,437075,cached
6,ch,394701,non-CTW,/home/msuarez/Escritorio/self_similarity/main_...,394701,cached


,n_nodes,n_pairs,n_successes,n_failures,success_ratio,avg_stretch,std_stretch,avg_topo_stretch,std_topo_stretch,label,...,edgelist_file,-_nb._vertices,-_beta,-_mu,-_radius_s1,-_radius_h2,-_kappa_min,edgelist_filename,rootname_output,manifestacion
0,178,31506,30978,528,0.983241,49.301612,11.659408,1.023396,0.108963,nat_CTW,...,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,178,36.6836,0.0353607,28.3296,28.4002,0.0330263,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,nat
1,246,60270,57256,3014,0.949992,48.477780,14.236101,1.022527,0.103639,9n_CTW,...,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,246,30.5728,0.0283465,39.1521,24.534,0.113997,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,/home/rcmrk/Escritorio/github/TFM_Manuel_Suare...,9n
2,289,83232,82538,694,0.991662,83.242715,19.909950,1.020736,0.091958,ch_CTW,...,/home/msuarez/Escritorio/self_similarity/main_...,289,27.704,0.086317,45.9958,35.1723,0.00495495,/home/msuarez/Escritorio/self_similarity/main_...,/home/msuarez/Escritorio/self_similarity/main_...,ch
3,313,97656,93267,4389,0.955057,41.499699,9.053070,1.017459,0.089554,ch_CTW_sensitivity,...,/data/394718.edge,313,28.5716,0.0818232,49.8155,20.8376,0.190698,/data/394718.edge,/data/394718,ch
4,18,306,306,0,1.000000,18.053186,7.827318,1.022876,0.093466,nat_non-CTW,...,/home/msuarez/Escritorio/self_similarity/main_...,18,4.8896,0.149883,2.86479,8.63735,0.713506,/home/msuarez/Escritorio/self_similarity/main_...,/home/msuarez/Escritorio/self_similarity/main_...,nat
5,181,32580,32408,172,0.994721,42.606934,12.118231,1.018067,0.089707,9n_non-CTW,...,/home/msuarez/Escritorio/self_similarity/main_...,181,4.39369,0.0439996,28.807,22.2498,0.138932,/home/msuarez/Escritorio/self_similarity/main_...,/home/msuarez/Escritorio/self_similarity/main_...,9n
6,151,22650,19776,2874,0.873113,34.812479,11.007618,1.007585,0.049337,ch_non-CTW,...,/home/msuarez/Escritorio/self_similarity/main_...,151,26.4109,0.134107,24.0324,14.5322,0.500467,/home/msuarez/Escritorio/self_similarity/main_...,/home/msuarez/Escritorio/self_similarity/main_...,ch


In [6]:
# 6. Respuestas formales a revisores
from generate_responses import build_response_matrix
responses = build_response_matrix()
display(responses)


,reviewer,topic,status,evidence,response_en,manuscript_change
0,R1,Fig. 2 styling,addressed,figures/fig2_combined_local_time.png,We thank the referee for these suggestions. Fi...,Update Fig. 2 caption; replace UTC labels with...
1,R1,Local time vs UTC,addressed,figures/fig2_*_local_time.png,All temporal axes and CTW annotations are now ...,Replace 'All times are given in UTC' in Fig. 2...
2,R1,Moving average purpose,addressed,results/smoothing_*.csv,The moving average is used to smooth sampling ...,Clarify smoothing purpose in Methods and SI.
3,R1,Fixed groups / modularity,addressed,results/community_fixed_*.csv,We added analyses with the Louvain partition f...,Add paragraph + figure in SI.
4,R1/R2,Navigability / diffusion claims,addressed,results/navigability_summary.csv; figures/navi...,We computed hyperbolic greedy-routing success ...,Replace TO BE DONE placeholders; moderate Disc...
5,R2,epsilon notation,addressed,results/epsilon_ctw_summary.csv,"We unified notation to epsilon^2_cco, epsilon^...",Remove mixed epsilon_c / epsilon_cco notation;...
6,R2,Clustering epsilon in Fig. 3 / SI.3,addressed,results/epsilon_ctw_summary.csv; figures/epsil...,We now report epsilon^2_cco explicitly for the...,Annotate Fig. 3/SI.3 panels with epsilon^2_cco...
7,R2,Non-CTW hyperbolic embedding,addressed,results/nonctw_selection.csv; results/embeddin...,We added non-CTW D-Mercator embeddings selecte...,Fix SI Sec. III window descriptions; add non-C...
8,R2,Null models,addressed,results/null_model_*.csv,We compared CTW metrics with degree-preserving...,Add null-model paragraph in Discussion/SI.
9,R2,Savitzky-Golay filter,addressed,results/smoothing_correlations_*.csv,"Savitzky-Golay smoothing (window 5, order 2) y...",Brief note in SI sensitivity section.


In [7]:
# Inventario de artefactos generados
from pathlib import Path
for folder in ['figures', 'results', 'embeddings']:
    p = REVIEW_ROOT / folder
    if p.exists():
        print(f'\n== {folder} ==')
        for f in sorted(p.rglob('*')):
            if f.is_file():
                print(f.relative_to(REVIEW_ROOT))



== figures ==
figures/community_fixed_9n.png
figures/community_fixed_ch.png
figures/community_fixed_nat.png
figures/embedding_pconn_9n_CTW_437037.png
figures/embedding_pconn_9n_non-CTW_437075.png
figures/embedding_pconn_ch_CTW_394717.png
figures/embedding_pconn_ch_CTW_sensitivity_394718.png
figures/embedding_pconn_ch_non-CTW_394701.png
figures/embedding_pconn_nat_CTW_429624.png
figures/embedding_pconn_nat_non-CTW_429600.png
figures/epsilon_collapse_9n.png
figures/epsilon_collapse_ch.png
figures/epsilon_collapse_nat.png
figures/fig2_9n_local_time.png
figures/fig2_ch_local_time.png
figures/fig2_combined_local_time.png
figures/fig2_nat_local_time.png
figures/navigability_ctw_vs_nonctw.png
figures/null_model_9n.png
figures/null_model_ch.png
figures/null_model_nat.png
figures/smoothing_9n.png
figures/smoothing_ch.png
figures/smoothing_nat.png

== results ==
results/community_fixed_9n.csv
results/community_fixed_ch.csv
results/community_fixed_nat.csv
results/embedding_inventory.csv
results/